In [ ]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [ ]:
# ============================================================
# RUNTIME BENCHMARK LOGGER
# ============================================================

from time import perf_counter
from datetime import datetime
from pathlib import Path
import pandas as pd
import os

RUNTIME_FILE = Path("FINAL_experimental_runtime_summary.csv")

if RUNTIME_FILE.exists():
    RUNTIME_LOG = pd.read_csv(RUNTIME_FILE).to_dict("records")
else:
    RUNTIME_LOG = []


def start_timer(experiment_name):
    print("\n" + "=" * 75)
    print(f"STARTING: {experiment_name}")
    print("Clock time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 75)

    return perf_counter()


def stop_timer(experiment_name, start_time):
    elapsed_sec = perf_counter() - start_time

    record = {
        "experiment": experiment_name,
        "runtime_seconds": elapsed_sec,
        "runtime_minutes": elapsed_sec / 60.0,
        "runtime_hours": elapsed_sec / 3600.0,
    }

    RUNTIME_LOG.append(record)

    pd.DataFrame(RUNTIME_LOG).to_csv(
        RUNTIME_FILE,
        index=False
    )

    print("\n" + "-" * 75)
    print(f"FINISHED: {experiment_name}")
    print(f"Runtime: {elapsed_sec:.2f} seconds")
    print(f"Runtime: {elapsed_sec / 60.0:.4f} minutes")
    print("-" * 75)

    return elapsed_sec

In [ ]:
diag_train = pd.read_csv("../Data/Raw/triage_train_diagnostics.csv")
clinical_slice_candidates = pd.read_csv("../Data/Raw/clinical_slice_candidates.csv")

print(diag_train.shape)
print(clinical_slice_candidates.shape)
diag_train.head()

Recreate the feature set for error-guided rule learning

In [ ]:
t_runtime_error = start_timer(
    "BASiC error-guided candidate discovery"
)
rule_feature_cols = [
    # complaint / symptom indicators
    "abdominal_pain", "chest_pain", "back_pain", "headache", "cough",
    "dizziness", "arm_pain", "breast_pain", "dental_pain", "diarrhea",
    "ear_pain", "eye_pain", "fever", "pelvic_pain", "unclassified",
    
    # comorbidity indicators (optional but interpretable)
    "ckd", "dm", "htn", "chd", "asthma", "bronchitis",
    "glaucoma", "hepatitis", "paralysis", "pneumonia", "sicklecell",
    
    # operational / missingness signals
    "arrival_mode_missing",
    "hr_missing", "pulse_rate_missing", "bp_systolic_missing",
    "bp_diastole_missing", "respiratory_missing", "temperature_missing", "spo2_missing",
    "vitals_missing_count",
    "heart_rate_disagreement_flag",
    
    # physiologic flags
    "tachycardia_flag", "hypotension_flag", "tachypnea_flag",
    "hypoxia_flag", "fever_temp_flag", "hypothermia_flag",
]

In [ ]:
def add_physiology_flags(df):
    df = df.copy()

    df["tachycardia_flag"] = (
        df["heart_rate_final"].notna() & (df["heart_rate_final"] > 100)
    ).astype(int)

    df["hypotension_flag"] = (
        df["bp_systolic"].notna() & (df["bp_systolic"] < 90)
    ).astype(int)

    df["tachypnea_flag"] = (
        df["respiratory"].notna() & (df["respiratory"] > 20)
    ).astype(int)

    df["hypoxia_flag"] = (
        df["spo2"].notna() & (df["spo2"] < 95)
    ).astype(int)

    df["fever_temp_flag"] = (
        df["temperature"].notna() & (df["temperature"] >= 38.0)
    ).astype(int)

    df["hypothermia_flag"] = (
        df["temperature"].notna() & (df["temperature"] < 36.0)
    ).astype(int)

    return df

In [ ]:
diag_train = add_physiology_flags(diag_train)
rule_df = add_physiology_flags(rule_df)

Basic cleaning for rule features

In [ ]:
rule_df = diag_train.copy()

for col in rule_feature_cols:
    if col in rule_df.columns:
        rule_df[col] = rule_df[col].fillna(0)

X_rule = rule_df[rule_feature_cols].copy()

print(X_rule.shape)
X_rule.head()

Define three error targets

In [ ]:
# already computed in Notebook 2, but ensure they exist
print(rule_df["high_loss_top10"].value_counts(normalize=True))
print(rule_df["false_negative_05"].value_counts(normalize=True))
print(rule_df["high_conf_contradiction"].value_counts(normalize=True))

Helper function to fit shallow tree and extract textual rules

In [ ]:
def fit_shallow_tree(X, y, max_depth=3, min_samples_leaf=500, random_state=42):
    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state
    )
    tree.fit(X, y)
    return tree

def print_tree_rules(tree, feature_names):
    rules = export_text(tree, feature_names=list(feature_names))
    print(rules)

Fit shallow tree for HIGH-LOSS pockets

In [ ]:
y_high_loss = rule_df["high_loss_top10"].astype(int)

tree_high_loss = fit_shallow_tree(
    X_rule, y_high_loss,
    max_depth=3,
    min_samples_leaf=1000,   # because dataset is large
    random_state=42
)

print_tree_rules(tree_high_loss, X_rule.columns)

Fit shallow tree for FALSE-NEGATIVE pockets

In [ ]:
y_fn = rule_df["false_negative_05"].astype(int)

tree_fn = fit_shallow_tree(
    X_rule, y_fn,
    max_depth=3,
    min_samples_leaf=1000,
    random_state=42
)

print_tree_rules(tree_fn, X_rule.columns)

Fit shallow tree for HIGH-CONFIDENCE CONTRADICTIONS

In [ ]:
y_hcc = rule_df["high_conf_contradiction"].astype(int)

tree_hcc = fit_shallow_tree(
    X_rule, y_hcc,
    max_depth=3,
    min_samples_leaf=1000,
    random_state=42
)

print_tree_rules(tree_hcc, X_rule.columns)

Create a manual rule library from tree findings

In [ ]:
error_slice_defs = {
    # Examples — replace/refine after reading your trees
    "err_vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
    "err_spo2_missing_and_unclassified": lambda d: (d["spo2_missing"] == 1) & (d["unclassified"] == 1),
    "err_hr_missing_and_abdominal_pain": lambda d: (d["hr_missing"] == 1) & (d["abdominal_pain"] == 1),
    "err_pneumonia_and_tachypnea": lambda d: (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1),
    "err_ckd_and_spo2_missing": lambda d: (d["ckd"] == 1) & (d["spo2_missing"] == 1),
    "err_chest_pain_and_tachycardia": lambda d: (d["chest_pain"] == 1) & (d["tachycardia_flag"] == 1),
    "err_unclassified_and_missingness": lambda d: (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 2),
    "err_hypoxia_and_arrival_missing": lambda d: (d["hypoxia_flag"] == 1) & (d["arrival_mode_missing"] == 1),
}

Reuse the slice metric function

In [ ]:
def slice_metrics(data, mask, y_col="y", p_col="oof_pred_proba", threshold=0.5):
    sub = data.loc[mask].copy()
    n = len(sub)
    
    if n == 0:
        return None
    
    y = sub[y_col].values
    p = sub[p_col].values
    y_pred = (p >= threshold).astype(int)
    
    auroc = np.nan
    if len(np.unique(y)) > 1:
        auroc = roc_auc_score(y, p)
    
    auprc = average_precision_score(y, p)
    
    tn = fp = fn = tp = 0
    if len(np.unique(y)) > 1:
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    else:
        if y[0] == 1:
            tp = (y_pred == 1).sum()
            fn = (y_pred == 0).sum()
        else:
            tn = (y_pred == 0).sum()
            fp = (y_pred == 1).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    return {
        "n": n,
        "support_pct": 100 * n / len(data),
        "event_rate": y.mean(),
        "AUROC": auroc,
        "AUPRC": auprc,
        "Recall@0.5": recall,
        "Precision@0.5": precision,
        "avg_logloss": sub["oof_logloss"].mean(),
        "high_loss_rate": sub["high_loss_top10"].mean(),
        "high_conf_contradiction_rate": sub["high_conf_contradiction"].mean(),
        "fn_rate": sub["false_negative_05"].mean()
    }

Compute error-guided slice table

In [ ]:
global_metrics = slice_metrics(rule_df, rule_df.index == rule_df.index)

rows = []

for slice_name, fn in error_slice_defs.items():
    mask = fn(rule_df)
    m = slice_metrics(rule_df, mask)
    
    if m is not None:
        m["slice_name"] = slice_name
        m["generator"] = "error_guided"
        m["auprc_gap_vs_global"] = m["AUPRC"] - global_metrics["AUPRC"]
        m["recall_gap_vs_global"] = m["Recall@0.5"] - global_metrics["Recall@0.5"]
        m["logloss_lift_vs_global"] = m["avg_logloss"] - global_metrics["avg_logloss"]
        rows.append(m)

error_slice_table = pd.DataFrame(rows)

error_slice_table = error_slice_table[
    [
        "slice_name", "generator", "n", "support_pct", "event_rate",
        "AUROC", "AUPRC", "Recall@0.5", "Precision@0.5",
        "avg_logloss", "high_loss_rate", "high_conf_contradiction_rate", "fn_rate",
        "auprc_gap_vs_global", "recall_gap_vs_global", "logloss_lift_vs_global"
    ]
]

error_slice_table.sort_values("logloss_lift_vs_global", ascending=False)

First-pass support filter

In [ ]:
min_support_pct = 0.5

error_slice_supported = error_slice_table[error_slice_table["support_pct"] >= min_support_pct].copy()

print("Supported error-guided slices:", len(error_slice_supported))
error_slice_supported.sort_values("logloss_lift_vs_global", ascending=False)

Rank error-guided slices

In [ ]:
error_slice_supported["priority_score"] = (
    2.0 * error_slice_supported["logloss_lift_vs_global"].fillna(0) +
    2.0 * error_slice_supported["fn_rate"].fillna(0) +
    1.5 * error_slice_supported["high_conf_contradiction_rate"].fillna(0) +
    1.0 * error_slice_supported["high_loss_rate"].fillna(0)
)

error_slice_ranked = error_slice_supported.sort_values("priority_score", ascending=False)

error_slice_ranked

Tag error slice types

In [ ]:
def tag_error_slice_type(name):
    if any(x in name for x in ["missing", "arrival", "vitals"]):
        return "operational_or_missingness"
    elif any(x in name for x in ["tachy", "hypo", "pneumonia", "ckd", "chest", "abdominal"]):
        return "clinical_or_mixed"
    else:
        return "mixed"

error_slice_ranked["slice_type"] = error_slice_ranked["slice_name"].apply(tag_error_slice_type)
error_slice_ranked

Save error-guided slice candidates

In [ ]:
error_slice_ranked.to_csv("error_guided_slice_candidates.csv", index=False)
print("Saved error_guided_slice_candidates.csv")
stop_timer(
    "BASiC error-guided candidate discovery",
    t_runtime_error
)